# 闭包、装饰器、迭代器与生成器

## Goal

理解函数是一等对象、闭包如何保存状态、装饰器如何包装函数，以及生成器为何是惰性计算。

## Setup

In [1]:
from functools import wraps
from itertools import islice

## Steps

### 1. 函数是一等对象

函数可以赋给变量、作为参数传入，也可以作为返回值。

In [2]:
def add(left, right):
    return left + right


operation = add
print("函数对象:", operation.__name__)
print("调用结果:", operation(2, 3))

函数对象: add
调用结果: 5


### 2. 闭包保存外层状态

In [3]:
def counter_factory(start=0):
    count = start

    def increment(step=1):
        nonlocal count
        count += step
        return count

    return increment


counter_a = counter_factory()
counter_b = counter_factory(100)
print("A:", counter_a(), counter_a(5))
print("B:", counter_b(), counter_b())

A: 1 6
B: 101 102


### 3. 装饰器增加横切功能

In [4]:
def trace(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print(f"调用 {func.__name__}{args}")
        result = func(*args, **kwargs)
        print(f"返回 {result!r}")
        return result

    return wrapper


@trace
def multiply(left, right):
    return left * right


product = multiply(6, 7)
print("函数名仍被保留:", multiply.__name__)

调用 multiply(6, 7)
返回 42
函数名仍被保留: multiply


### 4. 生成器按需计算

调用生成器函数时，函数体尚未执行；每次 `next()` 才推进到下一个 `yield`。

In [5]:
def fibonacci():
    left, right = 0, 1
    while True:
        yield left
        left, right = right, left + right


sequence = fibonacci()
print("生成器对象:", sequence)
print("前 8 项:", list(islice(sequence, 8)))

生成器对象: <generator object fibonacci at 0x74ebd5614110>
前 8 项: [0, 1, 1, 2, 3, 5, 8, 13]


### 5. 自定义迭代器协议

In [6]:
class Countdown:
    def __init__(self, start):
        self.current = start

    def __iter__(self):
        return self

    def __next__(self):
        if self.current == 0:
            raise StopIteration
        value = self.current
        self.current -= 1
        return value


print("倒计时:", list(Countdown(5)))

倒计时: [5, 4, 3, 2, 1]


## Checks

In [7]:
assert product == 42
assert counter_a() == 7
assert list(Countdown(3)) == [3, 2, 1]
assert list(islice(fibonacci(), 7)) == [0, 1, 1, 2, 3, 5, 8]
print("所有函数与迭代协议检查通过。")

所有函数与迭代协议检查通过。


## Next Steps

1. 给 `trace` 增加耗时统计。
2. 写一个可以重复迭代的 `CountdownIterable`，让 `__iter__` 返回新的迭代器。
3. 比较列表推导式和生成器表达式的内存占用。